In [ ]:
import pickle
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

In [ ]:
AMINO_ACID_CLASSES = {
    "polarity": {
        "hydrophobic": ["A", "V", "L", "I", "M", "F", "W", "P", "G"],
        "polar": ["S", "T", "Y", "C", "N", "Q"],
        "charged": ["K", "R", "H", "D", "E"],
    },

    "charge": {
        "positive": ["K", "R", "H"],
        "negative": ["D", "E"],
        "neutral": [
            "A", "V", "L", "I", "M", "F", "W", "P", "G",
            "S", "T", "Y", "C", "N", "Q"
        ],
    },

    "chemical_type": {
        "aliphatic": ["G", "A", "V", "L", "I"],
        "aromatic": ["F", "Y", "W"],
        "hydroxyl": ["S", "T", "Y"],
        "acidic": ["D", "E"],
        "amide": ["N", "Q"],
        "basic": ["K", "R", "H"],
        "sulfur": ["C", "M"],
        "imino": ["P"],
        # "other": [],  # placeholder (no gaps)
    },

    "essentiality": {
        "essential": ["F", "V", "T", "W", "M", "L", "I", "K", "H"],
        "non_essential": ["A", "N", "D", "E", "Q", "G", "P", "S", "C", "Y"],
    },

    "aromaticity": {
        "aromatic": ["F", "Y", "W"],
        "non_aromatic": [
            "A", "R", "N", "D", "C", "E", "Q", "G", "H", "I",
            "L", "K", "M", "P", "S", "T", "V"
        ],
    },

    "side_chain_size": {
        "small": ["G", "A", "S", "T", "P"],
        "medium": ["C", "N", "D", "Q", "E"],
        "large": ["V", "L", "I", "M", "F", "Y", "W", "K", "R", "H"],
    },
}

GRAM_POSITIVE = [
    "S. aureus ATCC 12600",
    "S. aureus (ATCC BAA-1556) - MRSA",
    "vancomycin-resistant E. faecalis ATCC 700802",
    "vancomycin-resistant E. faecium ATCC 700221",
    "L. monocytogenes ATCC 19111 (BEIRES NR-106)",
    "C. aerofaciens ATCC25986",
    "C. scindens ATCC35704",
    "C. spiroforme ATCC29900",
    "E. rectale ATCC33656",
    "C. symbiosum",
    "R. obeum",
    "R. torques",
]

GRAM_NEGATIVE = [
    "A. baumannii ATCC 19606",
    "E. coli ATCC 11775",
    "E. coli AIG221",
    "E. coli AIG222",
    "K. pneumoniae ATCC 13883",
    "P. aeruginosa PA01",
    "P. aeruginosa PA14",
    "E. coli Nissle",
    "Salmonella enterica ATCC 9150 (BEIRES NR-515)",
    "Salmonella enterica (BEIRES NR-170)",
    "Salmonella enterica ATCC 9150 (BEIRES NR-174)",

    "A. muciniphila ATCC BAA-835",
    "B. fragilis ATCC25285",
    "B. vulgatus ATCC8482",
    "B. thetaiotaomicron ATCC29148",
    "B. thetaiotaomicron Complemmented",
    "B. thetaiotaomicron Mutant",
    "B. uniformis ATCC8492",
    "B. eggerthi ATCC27754",
    "B. ovatus ATCC8483",
    "P. distasonis ATCC8503",
    "P. copri DSMZ18205",
]

In [ ]:
ALL_AA = sorted("ACDEFGHIKLMNPQRSTVWY")

In [ ]:
import numpy as np
import pandas as pd


def aggregate_mutation_dict(
    data_dict: dict,
    value_key: str,
    subset: list[str],      # REQUIRED
):
    """
    Aggregates mutation dictionaries of the structure:
        { (from_aa, to_aa): { value_key: { variable_name: list_of_values } } }

    Parameters
    ----------
    data_dict : dict
        Full mutation dictionary.
    value_key : str
        The inner key to aggregate over (e.g. "diff").
    subset : list of str
        REQUIRED list of variable names to include (e.g. species names).

    Returns
    -------
    df_mean, df_std, df_n : pivot tables
    """

    rows = []

    for (aa1, aa2), record in data_dict.items():
        if value_key not in record:
            continue

        all_values = []
        value_dict = record[value_key]

        # include only selected variables
        for var, vals in value_dict.items():
            if var in subset:
                all_values.extend(vals)

        if len(all_values) == 0:
            mean = np.nan
            std = np.nan
            n = 0
        else:
            arr = np.array(all_values, dtype=float)
            mean = float(np.mean(arr))
            std = float(np.std(arr))
            n = len(arr)

        rows.append({
            "from_aa": aa1,
            "to_aa": aa2,
            "mean": mean,
            "std": std,
            "n": n,
        })

    df = pd.DataFrame(rows)

    df_mean = df.pivot(index="from_aa", columns="to_aa", values="mean")
    df_std  = df.pivot(index="from_aa", columns="to_aa", values="std")
    df_n    = df.pivot(index="from_aa", columns="to_aa", values="n")

    return df_mean, df_std, df_n


---

In [ ]:
diff_values_path = Path('/home/pszmk/pep-compass/results/mutation_analysis/hydramp_x_ampsphere/direction_threshold=0.001_token_threshold=0.1_jacobian_mode=approx_jacobian_eps=0.05/ngram1/diff_values.csv')
diff_df = pd.read_csv(diff_values_path)

In [ ]:
from scripts.mutation_analysis.analysis import compute_mutation_statistics_from_df, compute_ranks_for_single_sample
from scripts.mutation_analysis.mutations import compute_aggregate_mutation_counter

In [ ]:
diff_df.head()

#### 1-grams

In [ ]:
aggregate_counter_ngram1 = compute_aggregate_mutation_counter(diff_df=diff_df,
parent_col='parent',
mutant_col='mutant',
ngram_size=1,
allow_ngrams_overlap=True)

mutation_statistics_ngram1 = compute_mutation_statistics_from_df(diff_df=diff_df,
aggregate_counter=aggregate_counter_ngram1,
parent_col='parent',
mutant_col='mutant',
value_cols=[f"{b}_log2_diff" for b in GRAM_POSITIVE + GRAM_NEGATIVE],
ngram_size=1,
allow_ngrams_overlap=True)

In [ ]:
gp_transition_mean, gp_transition_std, gp_transition_n = aggregate_mutation_dict(mutation_statistics_ngram1, value_key='diff', subset=GRAM_POSITIVE)
gn_transition_mean, gn_transition_std, gn_transition_n = aggregate_mutation_dict(mutation_statistics_ngram1, value_key='diff', subset=GRAM_NEGATIVE)

---

In [ ]:
transition_counts_statistics_path = Path('/home/pszmk/pep-compass/results/mutation_analysis/hydramp_x_ampsphere/direction_threshold=0.001_token_threshold=0.1_jacobian_mode=approx_jacobian_eps=0.05/ngram1/transition_counts_statistics.pkl')

with open(transition_counts_statistics_path, "rb") as f:
    transition_counts_statistics = pickle.load(f)

In [ ]:
transition_counts_statistics

In [ ]:
transition_counts_statistics[('E', 'C')]['diff']['A. baumannii ATCC 19606_log2']

#### Example on some bacteria

In [ ]:
transition_counts_statistics_mean, transition_counts_statistics_std, transition_counts_statistics_n = aggregate_mutation_dict(transition_counts_statistics, value_key='diff', subset=['A. baumannii ATCC 19606_log2'])

In [ ]:
transition_counts_statistics_mean

In [ ]:
transition_counts_statistics_std

In [ ]:
transition_counts_statistics_n

In [ ]:
np.nansum(transition_counts_statistics_n)

In [ ]:
group_colors = {
    "hydrophobic": "#F28E8E",
    "polar": "#8EC9F2",
    "charged": "#A9E68E",
}

In [ ]:
def aa_mutation_heatmap(
    df_mean: pd.DataFrame,
    aa_list: list[str],
    df_std: pd.DataFrame | None = None,
    df_n: pd.DataFrame | None = None,
    AA_group: dict[str, list[str]] | None = None,
    cmap: str = "magma",
    nan_color: str = "black",
    figsize=(9, 7),
    fontsize: int = 11,
    title: str = "",
    x_label: str = "To AA n-gram",
    y_label: str = "From AA n-gram",
    group_linewidth: float = 1.5,
    group_colors: dict[str, str] | None = None,

    # NEW: separate axis control
    x_bar_gap: float = 0.0,       # heatmap → X-axis color bar
    x_label_gap: float = 0.0,     # color bar → X-axis label
    y_bar_gap: float = 0.0,       # heatmap → Y-axis color bar
    y_label_gap: float = 0.0,     # color bar → Y-axis label
    group_label_color: str = "black",
):
    """
    Heatmap with fully symmetric and independently controlled
    X/Y axis group bars and group labels.
    """

    df = df_mean.copy()

    # === 1. Group sorting ===
    if AA_group is not None:
        sorted_aa = []
        group_sizes, group_names = [], []

        for gname, members in AA_group.items():
            present = [aa for aa in aa_list if aa in members]
            if present:
                sorted_aa.extend(present)
                group_sizes.append(len(present))
                group_names.append(gname)

        df = df.loc[sorted_aa, sorted_aa]
        if df_std is not None: df_std = df_std.loc[sorted_aa, sorted_aa]
        if df_n is not None:   df_n   = df_n.loc[sorted_aa, sorted_aa]
    else:
        sorted_aa = aa_list
        group_sizes, group_names = [], []


    # === 2. Annotation matrix ===
    annot = np.empty(df.shape, dtype=object)
    for i, r in enumerate(df.index):
        for j, c in enumerate(df.columns):
            if pd.isna(df.loc[r, c]):
                annot[i, j] = ""
                continue
            txt = f"{df.loc[r, c]:.2f}"
            if df_std is not None:
                sd = df_std.loc[r, c]
                if not pd.isna(sd):
                    txt += f"\n± {sd:.2f}"
            if df_n is not None:
                n = df_n.loc[r, c]
                if not pd.isna(n):
                    txt += f"\n(n={int(n)})"
            annot[i, j] = txt

    # === 3. NaN color ===
    base_cmap = plt.get_cmap(cmap).copy()
    base_cmap.set_bad(color=nan_color)
    mask = pd.isna(df)

    # === 4. Draw heatmap ===
    fig, ax = plt.subplots(figsize=figsize)

    sns.heatmap(
        df,
        cmap=base_cmap,
        annot=annot,
        fmt="",
        mask=mask,
        square=True,
        cbar=True,
        annot_kws={"fontsize": fontsize - 1},
        ax=ax,
    )

    ax.set_title(title, fontsize=fontsize + 2)
    ax.set_xlabel(x_label, fontsize=fontsize)
    ax.set_ylabel(y_label, fontsize=fontsize)
    ax.tick_params(axis="both", labelsize=fontsize)

    # === 5. Group boundaries + bars + labels (AXES COORDINATES) ===
    if AA_group is not None and group_sizes:

        boundaries = np.cumsum(group_sizes)
        mids = boundaries - np.array(group_sizes) / 2
        total = len(sorted_aa)

        # Convert mid positions from data coords → axes coords
        mids_axes = mids / total
        size_axes = np.array(group_sizes) / total

        bar_thickness_axes = 0.02  # thickness as fraction of axis

        # Separator lines (still in data coordinates)
        for b in boundaries[:-1]:
            ax.axhline(b, color="black", lw=group_linewidth)
            ax.axvline(b, color="black", lw=group_linewidth)

        # Draw bars + labels in AXES COORDINATES
        for mid_ax, gname, gsize_ax in zip(mids_axes, group_names, size_axes):

            col = group_colors.get(gname, "lightgray") if group_colors else "lightgray"

            # === Y-axis color bar ===
            ax.add_patch(plt.Rectangle(
                ( -y_bar_gap, mid_ax - gsize_ax/2 ),
                bar_thickness_axes,
                gsize_ax,
                transform=ax.transAxes,
                clip_on=False,
                color=col,
            ))

            # === Y-axis label ===
            ax.text(
                -y_bar_gap - bar_thickness_axes - y_label_gap,
                mid_ax,
                gname,
                ha="center",
                va="center",
                transform=ax.transAxes,
                rotation=90,
                fontsize=fontsize,
                color=group_label_color,
            )

            # === X-axis color bar (BOTTOM) ===
            ax.add_patch(plt.Rectangle(
                (mid_ax - gsize_ax/2, -x_bar_gap),
                gsize_ax,
                bar_thickness_axes,
                transform=ax.transAxes,
                clip_on=False,
                color=col,
            ))

            # === X-axis label (BOTTOM) ===
            ax.text(
                mid_ax,
                -x_bar_gap - bar_thickness_axes - x_label_gap,
                gname,
                ha="center",
                va="center",
                transform=ax.transAxes,
                fontsize=fontsize,
                color=group_label_color,
            )


    fig.tight_layout()
    return fig, ax


---

#### MIC

In [ ]:
aa_mutation_heatmap(
    df_mean=transition_counts_statistics_mean,
    aa_list=ALL_AA,
    df_std=transition_counts_statistics_std,
    df_n=transition_counts_statistics_n,
    AA_group=AMINO_ACID_CLASSES["polarity"],
    cmap="RdYlGn_r",
    nan_color="white",
    figsize=(24, 24),
    fontsize=11,
    title="",
    x_label="To AA n-gram",
    y_label="From AA n-gram",
    group_linewidth=4,
    group_colors=group_colors,
    x_bar_gap = 0.05,
    y_bar_gap = 0.05,
)

In [ ]:
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
import numpy as np
import pandas as pd


def mutation_mannwhitney_test(
    dict_a: dict,
    dict_b: dict,
    value_key: str,
    subset: list[str],                        # REQUIRED
    alternative: str = "two-sided",

    # NEW: kwargs
    mannwhitney_kwargs: dict | None = None,   # passed into mannwhitneyu
    multitest: bool = False,                  # whether to apply BH correction
    multitest_kwargs: dict | None = None,     # kwargs passed into multipletests

):
    """
    Performs Mann–Whitney U test for each mutation (aa1 -> aa2)
    comparing values from dict_a vs dict_b.

    Supports optional multiple testing correction (Benjamini–Hochberg).

    Parameters
    ----------
    dict_a, dict_b : dict
        Mutation dictionaries.
    value_key : str
        Inner key to extract values from.
    subset : list[str]
        REQUIRED list of variable names to include.
    alternative : str
        two-sided, less, greater (Mann–Whitney argument)
    mannwhitney_kwargs : dict
        Additional arguments forwarded to scipy.stats.mannwhitneyu.
    multitest : bool
        Whether to apply multiple testing correction (Benjamini–Hochberg).
    multitest_kwargs : dict
        Extra keyword arguments passed to statsmodels.multipletests().

    Returns
    -------
    df_p_raw : pivot table of raw p-values
    df_p_adj : pivot table of corrected p-values (NaN if multitest=False)
    df_u      : pivot table of U statistics
    df_n1     : pivot table of sample counts in dict_a
    df_n2     : pivot table of sample counts in dict_b
    df_effect : pivot table of median difference (median(a) - median(b))
    """

    if mannwhitney_kwargs is None:
        mannwhitney_kwargs = {}
    if multitest_kwargs is None:
        multitest_kwargs = {}

    rows = []

    # union of all mutation pairs
    all_keys = set(dict_a.keys()).union(dict_b.keys())

    for (aa1, aa2) in all_keys:

        def extract_values(rec):
            if rec is None or value_key not in rec:
                return []
            vals_out = []
            for var, vals in rec[value_key].items():
                if var in subset:
                    vals_out.extend(vals)
            return list(map(float, vals_out))

        vals_a = extract_values(dict_a.get((aa1, aa2)))
        vals_b = extract_values(dict_b.get((aa1, aa2)))

        # if one side has zero samples → cannot test
        if len(vals_a) == 0 or len(vals_b) == 0:
            u_stat = np.nan
            p_val = np.nan
            effect = np.nan
        else:
            try:
                u_stat, p_val = mannwhitneyu(
                    vals_a,
                    vals_b,
                    alternative=alternative,
                    **mannwhitney_kwargs
                )
                effect = np.median(vals_a) - np.median(vals_b)

            except Exception:
                u_stat = np.nan
                p_val = np.nan
                effect = np.nan

        rows.append({
            "from_aa": aa1,
            "to_aa": aa2,
            "p_raw": p_val,
            "u": u_stat,
            "n1": len(vals_a),
            "n2": len(vals_b),
            "effect": effect,
        })

    df = pd.DataFrame(rows)

    # --- Multiple testing correction ---
    if multitest:
        pvals = df["p_raw"].values
        _, p_adj, _, _ = multipletests(
            pvals,
            method="fdr_bh",       # Benjamini-Hochberg
            **multitest_kwargs
        )
        df["p_adj"] = p_adj
    else:
        df["p_adj"] = np.nan

    # --- Make pivot tables ---
    df_p_raw = df.pivot(index="from_aa", columns="to_aa", values="p_raw")
    df_p_adj = df.pivot(index="from_aa", columns="to_aa", values="p_adj")
    df_u     = df.pivot(index="from_aa", columns="to_aa", values="u")
    df_n1    = df.pivot(index="from_aa", columns="to_aa", values="n1")
    df_n2    = df.pivot(index="from_aa", columns="to_aa", values="n2")
    df_effect = df.pivot(index="from_aa", columns="to_aa", values="effect")

    return df_p_raw, df_p_adj, df_u, df_n1, df_n2, df_effect


In [ ]:
df_p_raw, df_p_adj, df_u, df_n1, df_n2, df_effect = mutation_mannwhitney_test(
    dict_a=transition_counts_statistics,
    dict_b=transition_counts_statistics,
    value_key="diff",
    subset=['A. baumannii ATCC 19606_log2'],     # REQUIRED
)


In [ ]:
df_u

---
## Comp with marginal

In [ ]:
from collections import defaultdict

# Marginalize over the second variable (sum all counts for each unique second element)
marginal_second = defaultdict(int)
for (first, second), count in transition_counts.items():
    marginal_second[second] += count

# Convert to regular dict and sort by value (descending)
marginal_second = dict(sorted(marginal_second.items(), key=lambda x: x[1], reverse=True))

marginal_second

---

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

In [ ]:
df = pd.DataFrame([
    {'from_aa': k[0], 'to_aa': k[1], 'count': v}
    for k, v in transition_counts.items()
])

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Build dataframe
df = pd.DataFrame([
    {'from_aa': k[0], 'to_aa': k[1], 'count': v}
    for k, v in transition_counts.items()
])

# Add a label string
df["label"] = df["from_aa"] + "→" + df["to_aa"]

# Sort
df_sorted = df.sort_values("count", ascending=False)

# Convert label to *ordered* categorical
df_sorted["label"] = pd.Categorical(
    df_sorted["label"],
    categories=df_sorted["label"],   # this keeps original order!
    ordered=True
)

plt.figure(figsize=(8, 48))
sns.barplot(
    data=df_sorted,
    x="count",
    y="label",     # now seaborn respects the order
    orient="h"
)
plt.xlabel("Count")
plt.ylabel("Mutation")
plt.title("Sorted mutation counts")
plt.tight_layout()
plt.show()
